# Chapter 42: 深度学习基础与量化应用 (Deep Learning Foundations & Quantitative Applications)

> **教学目标**：
> 1. 理解神经网络的结构归纳偏置（Structural Inductive Bias），掌握从手算链式法则到 NumPy 自动前向与反向传播的数学与编程实现；
> 2. 逐层拆解 CNN（局部连接与权重共享平移等变性）、RNN（状态递推与梯度积）、LSTM（恒定误差回路与门控）与 Transformer（缩放点积注意力、位置编码与严格因果掩码）；
> 3. 在同一时间分割、严格标签边界净化（Boundary Purging）与信息截止契约下，客观对比 4→8→1 MLP 与全局均值基准及正则化岭回归（Ridge）；
> 4. 深刻理解金融时间序列“低信噪比”、“小有效样本”与非平稳特征对深度模型的制约，并通过全流程组合账本揭示交易成本对高换手信号的实际损耗。


本 Notebook 使用离线教学构造数据，或明确标注的历史结果。新运行输出保存在单独的 `runs/` 子目录；这些内容不构成真实市场研究。


In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch42_deep_learning_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
import matplotlib.pyplot as plt

from quant_course.deep_learning import (
    forward,
    loss_and_grad,
    predict,
    fit_mlp,
    cnn_cross_correlation,
    rnn_states,
    lstm_step,
    causal_attention,
    sinusoidal_positional_encoding,
    generate_ch42_panel_data,
    split_ch42_panel,
)
from quant_course.ml_pipeline import backtest_predictions

print("Environment successfully initialized. Core functions imported.")



## 1. MLP 前向传播与反向传播梯度验证

考虑一个最小两层感知机结构 $2 \to 2 \to 1$：
- 输入 $\mathbf{X} \in \mathbb{R}^{B \times 2}$
- 隐藏层：$\mathbf{Z}_1 = \mathbf{X} \mathbf{W}_1 + \mathbf{b}_1 \in \mathbb{R}^{B \times 2}$，$\mathbf{A}_1 = \tanh(\mathbf{Z}_1)$
- 输出层：$\hat{\mathbf{Y}} = \mathbf{A}_1 \mathbf{W}_2 + \mathbf{b}_2 \in \mathbb{R}^{B \times 1}$
- 损失函数：均方误差（MSE）加 $L_2$ 正则化：
  $$\mathcal{L} = \frac{1}{B \cdot D_{\mathrm{out}}} \|\hat{\mathbf{Y}} - \mathbf{Y}\|_F^2 + \lambda (\|\mathbf{W}_1\|_F^2 + \|\mathbf{W}_2\|_F^2)$$

我们通过解析链式法则求出各参数梯度，并与中心差分（Central Difference）数值梯度核对比对：
$$\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \epsilon) - \mathcal{L}(\theta - \epsilon)}{2\epsilon}$$


In [ ]:
rng = np.random.default_rng(4200)
B, D_in, D_hidden, D_out = 3, 2, 2, 1
X_micro = rng.normal(size=(B, D_in))
y_micro = rng.normal(size=(B, D_out))

W1 = rng.normal(size=(D_in, D_hidden))
b1 = rng.normal(size=(1, D_hidden))
W2 = rng.normal(size=(D_hidden, D_out))
b2 = rng.normal(size=(1, D_out))
l2_reg = 0.01

loss, dW1, db1, dW2, db2 = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)

print(f"Total Loss (MSE + L2): {loss:.6f}")

# 中心差分对比
eps = 1e-6
param_dict = {"W1": (W1, dW1), "b1": (b1, db1), "W2": (W2, dW2), "b2": (b2, db2)}
grad_check_results = []

for name, (param, analytic_g) in param_dict.items():
    num_g = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = param[idx]
        param[idx] = orig + eps
        l_p, _, _, _, _ = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)
        param[idx] = orig - eps
        l_m, _, _, _, _ = loss_and_grad(X_micro, y_micro, W1, b1, W2, b2, l2_reg=l2_reg)
        param[idx] = orig
        num_g[idx] = (l_p - l_m) / (2.0 * eps)
        it.iternext()
    max_err = np.max(np.abs(analytic_g - num_g))
    grad_check_results.append({"Parameter": name, "Max Abs Error": f"{max_err:.2e}", "Status": "PASS" if max_err < 1e-7 else "FAIL"})

df_grad_check = pd.DataFrame(grad_check_results)
display(df_grad_check)


## 2. 深度架构核心算子微算例

本节逐项展示深度学习在量化建模中常用的四大算子，使用严格可手算的精简输入：
1. **CNN 一维有效互相关（Valid Cross-Correlation）**：输入 $[1, 2, 3, 4]$，卷积核 $[1, -1]$，得到局部差分特征 $[-1, -1, -1]$；
2. **Simple RNN 状态递推与梯度衰减**：$h_t = 0.5 h_{t-1} + x_t$，输入 $[1, 2, 0]$，状态序列为 $[1.0, 2.5, 1.25]$，长程梯度连乘因子为 $0.5^k$；
3. **现代 LSTM 单步门控与恒定误差回路（CEC）**：遗忘门、输入门、候选细胞、输出门交互；
4. **Transformer 因果缩放点积注意力与未来扰动屏蔽测试**：上三角掩码为 $-\infty$，保证未来输入扰动对历史输出完全无影响。


In [ ]:
# 1. CNN Valid Cross-Correlation
x_cnn = [1.0, 2.0, 3.0, 4.0]
w_cnn = [1.0, -1.0]
cnn_out = cnn_cross_correlation(x_cnn, w_cnn)
print(f"CNN Cross-Correlation output of [1,2,3,4] with [1,-1]: {cnn_out}")
assert np.allclose(cnn_out, [-1.0, -1.0, -1.0])

# 2. Simple RNN Recurrence
x_rnn = [1.0, 2.0, 0.0]
rnn_out = rnn_states(x_rnn, w_h=0.5, w_x=1.0, b=0.0)
print(f"RNN states for x=[1, 2, 0]: {rnn_out}")
assert np.allclose(rnn_out, [1.0, 2.5, 1.25])
print("Gradient factor across k=3 steps: (0.5)^3 =", 0.5**3)

# 3. Modern LSTM Step
x_t = np.array([0.5])
h_prev = np.array([0.1])
c_prev = np.array([0.2])
W_lstm = {k: np.array([[1.0]]) for k in ("f", "i", "c", "o")}
U_lstm = {k: np.array([[0.5]]) for k in ("f", "i", "c", "o")}
b_lstm = {k: np.array([0.0]) for k in ("f", "i", "c", "o")}
lstm_res = lstm_step(x_t, h_prev, c_prev, W_lstm, U_lstm, b_lstm)
print("LSTM Step Output:")
for k, v in lstm_res.items():
    print(f"  {k}: {np.round(v, 4)}")

# 4. Transformer Causal Attention & Perturbation Invariance Test
Q = np.array([[1.,0.],[0.,1.],[1.,1.]])
K = np.array([[1.,1.],[1.,0.],[0.,1.]])
V = np.array([[1.,2.],[3.,4.],[5.,6.]])
print("Position encoding (T=3,d=4):", sinusoidal_positional_encoding(3,4))

att_weights, att_out = causal_attention(Q, K, V, mask=True)
print("\nCausal Attention Weights (Masked):\n", np.round(att_weights, 4))
print("Attention output:", np.round(att_out,4))

# 扰动未来 token 2
Q_pert, K_pert, V_pert = Q.copy(), K.copy(), V.copy()
Q_pert[2] += 10.0
K_pert[2] -= 5.0
V_pert[2] += 8.0
_, att_out_pert = causal_attention(Q_pert, K_pert, V_pert, mask=True)

diff_history = np.max(np.abs(att_out[:2] - att_out_pert[:2]))
print(f"Max difference on earlier tokens (0 and 1) after perturbing future token 2: {diff_history:.2e}")
assert diff_history == 0.0, "Leakage detected!"


## 3. 构造资产面板数据与时间边界净化（Boundary Purging）

按照第42章冻结协议：
- 面板规模：$T=360$ 个工作日（时点带 UTC 时区），$N=12$ 个资产（A00..A11），特征数 $K=4$；
- 构造数据生成：弱非线性与独立高斯噪声（方差 $0.01^2$），低信噪比；
- 信号于 $D_k$ 16:00 UTC 收盘形成，次日 $D_{k+1}$ 收盘撮合成交（滞后 $\ell=1$），持有期 $h=1$ 至 $D_{k+2}$ 16:00 UTC 收盘结算；
- **边界净化契约**：
  - 训练集候选 $[0, 179]$：为确保标签结算时点 $D_{k+2} < D_{180}$，必须排除 $k=178, 179$；成熟训练样本为 $[0, 177]$（178个日期）；
  - 验证集候选 $[180, 239]$：为确保标签结算时点 $D_{k+2} < D_{240}$，必须排除 $k=238, 239$；成熟验证样本为 $[180, 237]$（58个日期）；
  - 测试集候选 $[240, 357]$：最后日期 $k=357$ 的标签于 $D_{359}$ 结算，完整落在360天面板内；成熟测试样本为 118 个日期。


In [ ]:
x, raw_return, close, dates, assets = generate_ch42_panel_data(seed=4201)
splits = split_ch42_panel(x, raw_return, dates, assets)

print(f"Panel Total Dates: {len(dates)}, Assets: {len(assets)}, Features: {x.shape[2]}")
print(f"Train mature dates: {len(splits.train_dates)} ({splits.train_dates[0]} to {splits.train_dates[-1]})")
print(f"Validation mature dates: {len(splits.val_dates)} ({splits.val_dates[0]} to {splits.val_dates[-1]})")
print(f"Test mature dates: {len(splits.test_dates)} ({splits.test_dates[0]} to {splits.test_dates[-1]})")
print("Purged boundary signal dates (preventing lookahead): [178, 179, 238, 239]")


## 4. 全模型同台实证基准测试

在相同的时间划分、信息截止与标准化口径下，比较以下三类模型：
1. **全局训练均值（Global Training Mean）**：1 参数单基准；
2. **岭回归（Ridge Regression）**：在验证集网格 $\alpha \in [0.1, 1.0, 10.0]$ 上调优；
3. **NumPy 4→8→1 tanh MLP**：49个参数，早停机制严格监控验证集 MSE，并在训练结束恢复验证集最优权重轮次。


In [ ]:
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

# 1. 特征与标签标准化（仅使用训练集均值和标准差）
mu_X = np.mean(splits.X_train, axis=0)
sigma_X = np.std(splits.X_train, axis=0)
mu_y = float(np.mean(splits.y_train))
sigma_y = float(np.std(splits.y_train))

X_train_s = (splits.X_train - mu_X) / sigma_X
y_train_s = (splits.y_train - mu_y) / sigma_y
X_val_s = (splits.X_val - mu_X) / sigma_X
y_val_s = (splits.y_val - mu_y) / sigma_y
X_test_s = (splits.X_test - mu_X) / sigma_X

# Model 1: Mean (1 parameter)
pred_mean = np.full_like(splits.y_test, mu_y)
mse_mean = float(np.mean((splits.y_test - pred_mean)**2))

# Model 2: Ridge
best_alpha, best_val_mse = None, float("inf")
for alpha in [0.1, 1.0, 10.0]:
    r = Ridge(alpha=alpha).fit(X_train_s, splits.y_train.ravel())
    v_mse = float(np.mean((splits.y_val.ravel() - r.predict(X_val_s))**2))
    if v_mse < best_val_mse:
        best_val_mse, best_alpha = v_mse, alpha

ridge_model = Ridge(alpha=best_alpha).fit(X_train_s, splits.y_train.ravel())
pred_ridge = ridge_model.predict(X_test_s).reshape(-1, 1)
mse_ridge = float(np.mean((splits.y_test - pred_ridge)**2))
r2_ridge = 1.0 - mse_ridge / mse_mean

# Model 3: MLP (4->8->1)
mlp_res = fit_mlp(
    X_train=X_train_s, y_train=y_train_s,
    X_val=X_val_s, y_val=y_val_s,
    hidden_dim=8, lr=0.01, batch_size=32, l2_reg=0.001,
    max_epochs=200, patience=15, seed_init=4202, seed_shuffle=4203
)
pred_mlp_s = predict(X_test_s, mlp_res.W1, mlp_res.b1, mlp_res.W2, mlp_res.b2)
pred_mlp = pred_mlp_s * sigma_y + mu_y
mse_mlp = float(np.mean((splits.y_test - pred_mlp)**2))
r2_mlp = 1.0 - mse_mlp / mse_mean

print(f"MLP Early stopping stopped at epoch {mlp_res.stopped_epoch}, restored best epoch {mlp_res.best_epoch}.")


## 5. 样本外量化评价与组合账本回测

对所有模型预测进行截面去均值（Dollar-Neutral），总敞口设为 1.0；常数预测不形成仓位（敞口为 0）。
调用 Chapter 29 撮合账本，模拟执行滞后 $\ell=1$、单边交易成本分别为 0（毛收益）和 10 bps（净收益）下的资金净值。


In [ ]:
# Daily Spearman IC
n_test_dates = len(splits.test_dates)
N = len(assets)
y_test_2d = splits.y_test.reshape(n_test_dates, N)
pred_ridge_2d = pred_ridge.reshape(n_test_dates, N)
pred_mlp_2d = pred_mlp.reshape(n_test_dates, N)

ic_ridge = [spearmanr(pred_ridge_2d[t], y_test_2d[t])[0] for t in range(n_test_dates)]
ic_mlp = [spearmanr(pred_mlp_2d[t], y_test_2d[t])[0] for t in range(n_test_dates)]

# Backtests
idx_test = pd.MultiIndex.from_product([splits.test_dates, assets], names=["signal_time", "asset"])
bt_mean_g = backtest_predictions(close, pd.Series(pred_mean.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_mean_n = backtest_predictions(close, pd.Series(pred_mean.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

bt_ridge_g = backtest_predictions(close, pd.Series(pred_ridge.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_ridge_n = backtest_predictions(close, pd.Series(pred_ridge.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

bt_mlp_g = backtest_predictions(close, pd.Series(pred_mlp.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.0)
bt_mlp_n = backtest_predictions(close, pd.Series(pred_mlp.ravel(), index=idx_test), rebalance_every=1, fee_rate=0.001)

# Summary DataFrame
summary_table = pd.DataFrame({
    "Parameters": [1, 5, 49],
    "Test MSE": [f"{mse_mean:.8f}", f"{mse_ridge:.8f}", f"{mse_mlp:.8f}"],
    "Test R^2": ["0.000000", f"{r2_ridge:.6f}", f"{r2_mlp:.6f}"],
    "Daily Spearman IC": ["NaN (Constant)", f"{np.mean(ic_ridge):.4f}", f"{np.mean(ic_mlp):.4f}"],
    "Gross Sharpe": ["NaN", f"{bt_ridge_g['metrics']['sharpe']:.3f}", f"{bt_mlp_g['metrics']['sharpe']:.3f}"],
    "Net Sharpe (10 bps)": ["NaN", f"{bt_ridge_n['metrics']['sharpe']:.3f}", f"{bt_mlp_n['metrics']['sharpe']:.3f}"],
    "Annual Turnover": [0.0, f"{bt_ridge_n['metrics']['turnover_ann']:.1f}", f"{bt_mlp_n['metrics']['turnover_ann']:.1f}"],
}, index=["Global Training Mean", "Ridge Regression", "NumPy MLP (4->8->1)"])

display(summary_table)


## 6. 本次构造运行生成的图

以下图全部来自当前数组、教学模型与账本，不读取旧保存图。


In [ ]:
# Use this cold run's computed arrays and fitted teaching model, not old saved figures.
def save_display(fig, filename):
    fig.tight_layout()
    fig.savefig(RUN_DIR / filename, dpi=150)
    display(Image(filename=str(RUN_DIR / filename)))
    plt.close(fig)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(mlp_res.train_loss_history, label="train")
ax.plot(mlp_res.val_loss_history, label="validation")
ax.axvline(mlp_res.best_epoch, linestyle="--", label="best epoch")
ax.set(xlabel="Epoch", ylabel="Scaled MSE", title="CONSTRUCTED MLP loss")
ax.legend(); save_display(fig, "loss_curve.png")

fig, axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax, name, pred in zip(axes,["Ridge","MLP"],[pred_ridge,pred_mlp]):
    ax.scatter(pred.ravel(),splits.y_test.ravel(),s=3,alpha=.3)
    ax.set(xlabel="Predicted simple return",ylabel="Realized simple return",title="CONSTRUCTED " + name)
save_display(fig,"predictions_vs_targets.png")

fig, axes = plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax, title, results in zip(axes,["Gross","Net 10 bp"],[(bt_mean_g,bt_ridge_g,bt_mlp_g),(bt_mean_n,bt_ridge_n,bt_mlp_n)]):
    for name, result in zip(["Mean","Ridge","MLP"], results):
        nav=result["ledger"]["nav"]
        ax.plot(nav.index,nav/nav.iloc[0],label=name)
    ax.set(xlabel="Date",ylabel="Wealth (initial 1)",title="CONSTRUCTED " + title)
    ax.legend(); ax.tick_params(axis="x",rotation=30)
save_display(fig,"cumulative_returns.png")

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(splits.test_dates,ic_ridge,label="Ridge")
ax.plot(splits.test_dates,ic_mlp,label="MLP")
ax.set(xlabel="Test date",ylabel="Daily Spearman IC",title="CONSTRUCTED daily cross-sectional IC")
ax.legend(); ax.tick_params(axis="x",rotation=30)
save_display(fig,"daily_ic_timeseries.png")

fig, ax = plt.subplots(figsize=(5,4))
im=ax.imshow(att_weights,vmin=0,vmax=1)
ax.set(xlabel="Key token",ylabel="Query token",title="CONSTRUCTED causal attention")
fig.colorbar(im,ax=ax); save_display(fig,"causal_attention_weights.png")


## 7. 量化深度学习的核心方法论启示

从本章构造实验和数学推导中，量化研究者可以总结出四条核心方法论认知：

1. **结构归纳偏置与参数有效性**：MLP 拥有 49 个参数，训练块有178个日期、2136行构造样本；日期数不等同独立样本量。在低信噪比金融环境中，缺乏结构归纳偏置的全连接网络容易过度拟合样本噪声，导致测试集 $R^2$ 弱于单参数全局均值基准。
2. **时间分割与严格边界净化**：当收益标签跨越多个日期（如 $D_k$ 信号预测 $D_{k+1}$ 至 $D_{k+2}$ 收益）时，必须严格排除跨越边界的数据。若不进行净化，标签将在训练集与验证集/测试集之间发生前瞻泄漏。
3. **因果掩码的局限性**：Transformer 的因果注意力掩码仅保证在给定当前输入序列下计算注意力权重时不向后引用，但如果底层特征在构造阶段就包含了未来收益或由于不当预处理引入了全局泄漏，因果掩码无法消除上游的前瞻漏洞。
4. **交易成本对净收益的决定性影响**：未扣除交易成本的毛 Sharpe（约 0.5–0.8），在 350+ 倍年化换手率与 10 bps 单边成本的磨损下，净 Sharpe 降至 -4.7 到 -5.5。量化策略必须始终基于撮合账本扣除交易摩擦评估真实表现。
